# Day 8d — Feature fusion with real GNN embeddings

Attempt 3 (`notebooks/08b_feature_fusion_classifier.ipynb`) fed the classifier a single flat category label (`ring_category`) derived from the graph — and it barely helped (ring features ranked last in importance). Research on this exact dataset pointed to why: production systems feed a classifier the GNN's **dense learned embedding** (a 128-number summary of each transaction and its graph neighborhood), not a coarse 4-value label.

This notebook tests that directly: same classifier, same tabular features, but **128 real embedding columns instead of 1 flat label**. The embeddings come from the tuned GNN already trained and evaluated (`temp/gnn_extract_embeddings.py` — holdout PR-AUC 0.6359, matching the earlier reported result exactly, confirming determinism).

In [1]:
import sys, json
sys.path.append('..')

import numpy as np
import pandas as pd
from sklearn.metrics import precision_score, recall_score, f1_score, average_precision_score, confusion_matrix

from src.data import load_merged_train
from src.features import get_feature_lists
from src.split import apply_locked_split
from src.model import build_xgb_pipeline
from src.cost import find_optimal_threshold, DEFAULT_REVIEW_COST

RANDOM_STATE = 42

## 1. Attach the GNN embeddings as ordinary numeric features

In [2]:
train_full = load_merged_train()

embeddings = np.load('../temp/gnn_embeddings.npy')
print('Embeddings shape:', embeddings.shape)
assert embeddings.shape[0] == len(train_full), 'Embedding row count must match train_full row count/order'

embedding_cols = [f'gnn_emb_{i}' for i in range(embeddings.shape[1])]
embedding_df = pd.DataFrame(embeddings, columns=embedding_cols, index=train_full.index)
train_full = pd.concat([train_full, embedding_df], axis=1)

numeric_features, categorical_features = get_feature_lists(train_full)
numeric_features = numeric_features + embedding_cols

print(f'{len(numeric_features)} numeric features (incl. {len(embedding_cols)} GNN embedding dims)')
print(f'{len(categorical_features)} categorical features')

Embeddings shape: (590540, 128)


516 numeric features (incl. 128 GNN embedding dims)
15 categorical features


## 2. Train and evaluate on the same locked holdout

In [3]:
train_df, holdout_df = apply_locked_split(train_full)

X_train = train_df[numeric_features + categorical_features]
y_train = train_df['isFraud']
X_holdout = holdout_df[numeric_features + categorical_features]
y_holdout = holdout_df['isFraud']
amounts_holdout = holdout_df['TransactionAmt']

scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()

model_v5 = build_xgb_pipeline(numeric_features, categorical_features, scale_pos_weight=scale_pos_weight, random_state=RANDOM_STATE)
model_v5.fit(X_train, y_train)
print('Fitted v5 (GNN-embedding-fusion) classifier.')

Fitted v5 (GNN-embedding-fusion) classifier.


In [4]:
y_proba_v5 = model_v5.predict_proba(X_holdout)[:, 1]

v5_threshold, v5_cost, _ = find_optimal_threshold(y_holdout, y_proba_v5, amounts_holdout, review_cost=DEFAULT_REVIEW_COST)

with open('../results/classifier_final_metrics.json') as f:
    locked_v2_metrics = json.load(f)
with open('../results/feature_fusion_metrics.json') as f:
    v3_metrics = json.load(f)
v2_cost = locked_v2_metrics['total_cost_rs']
v3_cost = v3_metrics['total_cost_rs']

print(f'v5 (GNN embeddings) cost-optimal threshold: {v5_threshold:.2f}')
print(f'v5 cost: Rs {v5_cost:,.0f}')
print(f'v2 (locked, no graph features) cost: Rs {v2_cost:,.0f}')
print(f'v3 (flat category label) cost: Rs {v3_cost:,.0f}')
print(f'v5 vs v2: Rs {v2_cost - v5_cost:,.0f} ({"v5 better" if v5_cost < v2_cost else "v2 better"})')
print(f'v5 vs v3: Rs {v3_cost - v5_cost:,.0f} ({"v5 better" if v5_cost < v3_cost else "v3 better"})')

v5 (GNN embeddings) cost-optimal threshold: 0.90
v5 cost: Rs 374,903
v2 (locked, no graph features) cost: Rs 337,421
v3 (flat category label) cost: Rs 333,872
v5 vs v2: Rs -37,482 (v2 better)
v5 vs v3: Rs -41,030 (v3 better)


In [5]:
y_pred_v5 = (y_proba_v5 >= v5_threshold).astype(int)
precision_v5 = precision_score(y_holdout, y_pred_v5)
recall_v5 = recall_score(y_holdout, y_pred_v5)
f1_v5 = f1_score(y_holdout, y_pred_v5)
pr_auc_v5 = average_precision_score(y_holdout, y_proba_v5)
cm_v5 = confusion_matrix(y_holdout, y_pred_v5)

print(f'Precision: {precision_v5:.4f} (v2 locked: {locked_v2_metrics["precision"]:.4f}, v3: {v3_metrics["precision"]:.4f})')
print(f'Recall:    {recall_v5:.4f} (v2 locked: {locked_v2_metrics["recall"]:.4f}, v3: {v3_metrics["recall"]:.4f})')
print(f'F1:        {f1_v5:.4f} (v2 locked: {locked_v2_metrics["f1"]:.4f}, v3: {v3_metrics["f1"]:.4f})')
print(f'PR-AUC:    {pr_auc_v5:.4f} (v2 locked: {locked_v2_metrics.get("pr_auc", float("nan")):.4f}, v3: {v3_metrics["pr_auc"]:.4f})')
print('\nConfusion matrix:')
print(cm_v5)

Precision: 0.8020 (v2 locked: 0.7582, v3: 0.8350)
Recall:    0.5911 (v2 locked: 0.6593, v3: 0.6136)
F1:        0.6806 (v2 locked: 0.7053, v3: 0.7074)
PR-AUC:    0.7081 (v2 locked: nan, v3: 0.7566)

Confusion matrix:
[[113372    603]
 [  1690   2443]]


## 3. Did the embeddings actually matter this time?

In [6]:
preprocessor_v5 = model_v5.named_steps['prep']
classifier_v5 = model_v5.named_steps['clf']
feature_names_v5 = preprocessor_v5.get_feature_names_out()

importances = pd.Series(classifier_v5.feature_importances_, index=feature_names_v5).sort_values(ascending=False)

emb_importances = importances[importances.index.str.contains('gnn_emb_')]
print(f'GNN embedding dims: {len(emb_importances)} total')
print(f'Best-ranked embedding dim: rank {importances.index.get_loc(emb_importances.index[0]) + 1} of {len(importances)} (importance={emb_importances.iloc[0]:.5f})')
print(f'Sum of all embedding-dim importances: {emb_importances.sum():.4f} (out of total 1.0 across all features)')
print(f'How many embedding dims land in the top 20 overall: {sum(1 for name in importances.head(20).index if "gnn_emb_" in name)}')

print('\nTop 15 features overall:')
print(importances.head(15))

GNN embedding dims: 128 total
Best-ranked embedding dim: rank 1 of 683 (importance=0.13996)
Sum of all embedding-dim importances: 0.5871 (out of total 1.0 across all features)
How many embedding dims land in the top 20 overall: 15

Top 15 features overall:
num__gnn_emb_127    0.139956
num__gnn_emb_9      0.057582
num__gnn_emb_45     0.037612
num__gnn_emb_54     0.030607
num__gnn_emb_93     0.021518
num__gnn_emb_29     0.013036
num__gnn_emb_40     0.012176
num__V151           0.011679
num__gnn_emb_5      0.010646
num__gnn_emb_92     0.010484
num__gnn_emb_0      0.010428
num__gnn_emb_72     0.008368
num__gnn_emb_55     0.008334
num__V118           0.007930
num__gnn_emb_33     0.007699
dtype: float32


## Save results

In [7]:
gnn_fusion_results = {
    'model': 'XGBoost v5 (128-dim GNN embedding fusion)',
    'threshold': v5_threshold,
    'precision': precision_v5,
    'recall': recall_v5,
    'f1': f1_v5,
    'pr_auc': pr_auc_v5,
    'confusion_matrix': cm_v5.tolist(),
    'total_cost_rs': v5_cost,
    'v2_locked_cost_rs': v2_cost,
    'v3_flat_label_cost_rs': v3_cost,
    'difference_vs_v2_rs': v2_cost - v5_cost,
    'difference_vs_v3_rs': v3_cost - v5_cost,
    'embedding_importance_sum': float(emb_importances.sum()),
    'embedding_dims_in_top_20': int(sum(1 for name in importances.head(20).index if 'gnn_emb_' in name)),
    'best_embedding_dim_rank': int(importances.index.get_loc(emb_importances.index[0]) + 1),
}

with open('../results/gnn_embedding_fusion_metrics.json', 'w') as f:
    json.dump(gnn_fusion_results, f, indent=2, default=str)

print('Saved to results/gnn_embedding_fusion_metrics.json')

Saved to results/gnn_embedding_fusion_metrics.json


## Takeaway

_Fill in after running: whether v5 beats v2 and v3 on cost, whether embedding dims rank meaningfully higher than the flat category label did (confirming or refuting the research prediction), and the final honest verdict on this fourth combination strategy._